# LeetCode 3386 — Button with Longest Push Time bằng Python

Link đề: https://leetcode.com/problems/button-with-longest-push-time/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Single Pass (một lượt duyệt), Tie-break (xử lý khi hoà)

Skeleton LeetCode:

```python
class Solution:
    def buttonWithLongestTime(self, events: List[List[int]]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án được kiểm chứng bằng code (đối chiếu bốn cách làm với một oracle độc lập trên nhiều input ngẫu nhiên đúng ràng buộc). Ví dụ tự dựng thêm được ghi rõ "tự dựng" — nếu cần khớp từng chữ thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một đứa trẻ bấm các nút trên bàn phím; `events[i] = [index, time]` nghĩa là nút `index` được bấm lúc `time`; thời gian "bấm" của một lần là **hiệu** giữa thời điểm này và thời điểm của lần bấm **ngay trước** (lần đầu thì là chính `time`). Trả về **chỉ số nút** có lần bấm **lâu nhất**; hoà thì chọn chỉ số **nhỏ nhất**.
2. Nắm các công cụ Python: duyệt danh sách các cặp `[index, time]` với unpack, biến `prev` giữ thời điểm trước, `zip(events, events[1:])` / `itertools.pairwise`, so sánh tuple, `max(..., key=...)`, và từ điển `dict` với `.get`.
3. Đi từ cách **lập danh sách thời gian rồi tìm lớn nhất** → **gom theo nút bằng dict** → **một lượt duyệt giữ `ans`, `t`** → **`max` với khoá `(thời gian, -chỉ số)`**.
4. Hiểu mấu chốt: chỉ cần so sánh từng **lần bấm** với "kỷ lục hiện tại"; khi **thời gian bằng nhau** thì chuyển sang nút có **chỉ số nhỏ hơn** (`d == t and index < ans`); đáp án là **chỉ số nút** (`events[k][0]`), **không** phải vị trí `k` trong danh sách.
5. Tránh các lỗi kinh điển: bỏ sót lần bấm đầu tiên, dùng `time` thay vì hiệu thời gian, quên luật hoà (trả nút xuất hiện sau cùng hay trước nhất thay vì chỉ số nhỏ nhất), trả vị trí `k` thay vì `index`, và **cộng dồn** thời gian của cùng một nút thay vì chỉ lấy lần bấm lâu nhất.

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `events` mô tả một dãy sự kiện: một đứa trẻ bấm một loạt nút trên bàn phím. Mỗi `events[i] = [index_i, time_i]` nghĩa là nút có chỉ số `index_i` được bấm tại thời điểm `time_i`.

- Mảng đã được **sắp xếp tăng dần theo `time`**.
- **Thời gian bấm** một nút là hiệu giữa thời điểm bấm nút đó và thời điểm bấm nút **liền trước**. Với nút đầu tiên, thời gian bấm chính là thời điểm nó được bấm.

Trả về **chỉ số (`index`)** của nút có thời gian bấm **lâu nhất**. Nếu có nhiều nút cùng lâu nhất, trả về nút có **chỉ số nhỏ nhất**.

**Ví dụ 1**

- Input: `events = [[1,2],[2,5],[3,9],[1,15]]`
- Output: `1`
- Giải thích: nút `1` bấm lúc `2` (thời gian `2`); nút `2` lúc `5` ⇒ `5 - 2 = 3`; nút `3` lúc `9` ⇒ `9 - 5 = 4`; nút `1` bấm lại lúc `15` ⇒ `15 - 9 = 6`. Lâu nhất là `6`, của nút `1`.

**Ví dụ 2**

- Input: `events = [[10,5],[1,7]]`
- Output: `10`
- Giải thích: nút `10` bấm lúc `5` (thời gian `5`); nút `1` lúc `7` ⇒ `7 - 5 = 2`. Lâu nhất là `5`, của nút `10`.

**Ràng buộc**

- `1 <= events.length <= 1000`
- `events[i] == [index_i, time_i]`
- `1 <= index_i, time_i <= 10^5`
- Dữ liệu được tạo sao cho `events` sắp xếp tăng dần theo `time_i`.

In [1]:
from typing import List

examples = [
    ([[1, 2], [2, 5], [3, 9], [1, 15]], 1),
    ([[10, 5], [1, 7]], 10),
]

def button_quick(events):
    """Bản nhanh để chạy thử ví dụ: lập danh sách (thời gian bấm, chỉ số nút), rồi chọn."""
    prev = 0
    best = None
    for idx, t in events:
        d = t - prev
        prev = t
        if best is None or d > best[0] or (d == best[0] and idx < best[1]):
            best = (d, idx)
    return best[1]

for events, expected in examples:
    print(f"events={events} -> {button_quick(events)} (kỳ vọng {expected})")
    assert button_quick(events) == expected

events=[[1, 2], [2, 5], [3, 9], [1, 15]] -> 1 (kỳ vọng 1)
events=[[10, 5], [1, 7]] -> 10 (kỳ vọng 10)


## 3. Kiến thức nền

### 3.1. `events` là danh sách các cặp, và "unpack"

Mỗi phần tử của `events` là một danh sách hai số `[index, time]`. Trong vòng `for`, ta có thể **unpack**: `for idx, t in events:` — Python tự tách mỗi cặp thành hai biến `idx` và `t`. Cách này dễ đọc hơn `event[0]`, `event[1]`.

In [2]:
events = [[1, 2], [2, 5], [3, 9], [1, 15]]
for event in events:
    print("cách 1:", event[0], event[1])

for idx, t in events:
    print("cách 2: nút", idx, "bấm lúc", t)

cách 1: 1 2
cách 1: 2 5
cách 1: 3 9
cách 1: 1 15
cách 2: nút 1 bấm lúc 2
cách 2: nút 2 bấm lúc 5
cách 2: nút 3 bấm lúc 9
cách 2: nút 1 bấm lúc 15


### 3.2. Thời gian bấm = hiệu hai thời điểm liên tiếp

Để tính thời gian của mỗi lần bấm, ta cần biết thời điểm của **lần bấm trước**. Giữ một biến `prev`: lúc đầu `prev = 0` (thời điểm "bắt đầu", để lần bấm đầu có thời gian đúng bằng `time` của nó); sau mỗi lần, cập nhật `prev = t`. Khi đó `d = t - prev`.

In [3]:
events = [[1, 2], [2, 5], [3, 9], [1, 15]]
prev = 0
for idx, t in events:
    d = t - prev
    print(f"nút {idx}: bấm lúc {t}, lần trước lúc {prev}, thời gian = {d}")
    prev = t

nút 1: bấm lúc 2, lần trước lúc 0, thời gian = 2
nút 2: bấm lúc 5, lần trước lúc 2, thời gian = 3
nút 3: bấm lúc 9, lần trước lúc 5, thời gian = 4
nút 1: bấm lúc 15, lần trước lúc 9, thời gian = 6


### 3.3. Cặp liền kề: `zip(events, events[1:])` và `pairwise`

Một cách khác: ghép mỗi sự kiện với sự kiện **ngay sau** nó. `zip(a, a[1:])` tạo các cặp `(a[0], a[1])`, `(a[1], a[2])`, ... Từ Python 3.10 có `itertools.pairwise(a)` làm đúng việc đó. Với cách này, lần bấm đầu tiên phải xử lý **riêng** (khởi tạo `ans`, `t` bằng chính sự kiện đầu).

In [4]:
from itertools import pairwise

events = [[1, 2], [2, 5], [3, 9], [1, 15]]
print(list(zip(events, events[1:])))
print(list(pairwise(events)))
for (_, t1), (idx, t2) in pairwise(events):
    print(f"nút {idx}: {t2} - {t1} = {t2 - t1}")

[([1, 2], [2, 5]), ([2, 5], [3, 9]), ([3, 9], [1, 15])]
[([1, 2], [2, 5]), ([2, 5], [3, 9]), ([3, 9], [1, 15])]
nút 2: 5 - 2 = 3
nút 3: 9 - 5 = 4
nút 1: 15 - 9 = 6


### 3.4. Luật hoà (tie-break): "lớn hơn" hay "bằng nhưng chỉ số nhỏ hơn"

Ta giữ "kỷ lục" `(t, ans)` = (thời gian lớn nhất, nút tương ứng). Với mỗi lần bấm `(d, idx)`:

- nếu `d > t`: kỷ lục mới;
- nếu `d == t` **và** `idx < ans`: cũng nhận (cùng thời gian nhưng chỉ số nhỏ hơn);
- ngược lại: bỏ qua.

Có thể viết gọn bằng **so sánh tuple**: `(d, -idx) > (t, -ans)`. Python so sánh tuple **từng phần tử từ trái sang phải**: trước hết `d`, nếu bằng nhau mới so `-idx` (chỉ số nhỏ hơn ⇒ `-idx` lớn hơn).

In [5]:
print((4, -1) > (4, -3))     # True : cùng thời gian 4, nút 1 (-1) tốt hơn nút 3 (-3)
print((5, -9) > (4, -1))     # True : thời gian 5 > 4, bất kể chỉ số
print((4, -3) > (4, -1))     # False

# Chuyển đổi: "d > t or (d == t and idx < ans)"  tương đương  "(d, -idx) > (t, -ans)"
for d, idx, t, ans in [(4, 1, 4, 3), (5, 9, 4, 1), (4, 3, 4, 1), (2, 1, 4, 3)]:
    a = d > t or (d == t and idx < ans)
    b = (d, -idx) > (t, -ans)
    print(f"d={d} idx={idx} t={t} ans={ans}: {a} {b}")
    assert a == b

True
True
False
d=4 idx=1 t=4 ans=3: True True
d=5 idx=9 t=4 ans=1: True True
d=4 idx=3 t=4 ans=1: False False
d=2 idx=1 t=4 ans=3: False False


### 3.5. `max(..., key=...)` và từ điển `dict`

- `max(iterable, key=hàm)` chọn phần tử có `hàm(phần tử)` lớn nhất (nếu hoà thì lấy phần tử **đầu tiên** gặp). Ta sẽ cho `key` là `(thời gian, -chỉ số)` để cả hai tiêu chí cùng được xét.
- `dict` (từ điển) lưu cặp `khóa: giá trị`. Ở bài này `best[idx]` = thời gian bấm lâu nhất của nút `idx`. Mẫu: `best[idx] = max(best.get(idx, 0), d)`: `.get(idx, 0)` cho `0` nếu `idx` chưa có.

In [6]:
best = {}
for idx, d in [(1, 2), (2, 3), (3, 4), (1, 6)]:
    best[idx] = max(best.get(idx, 0), d)
print(best)                               # {1: 6, 2: 3, 3: 4}

pairs = [(2, 1), (3, 2), (4, 3), (6, 1)]   # (thời gian, chỉ số)
print(max(pairs))                          # (6, 1)
print(max(pairs, key=lambda p: (p[0], -p[1])))   # (6, 1)
print(max([(4, 3), (4, 1), (4, 2)], key=lambda p: (p[0], -p[1])))   # (4, 1): hoà -> chỉ số nhỏ nhất

{1: 6, 2: 3, 3: 4}
(6, 1)
(6, 1)
(4, 1)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `events = [[1,2],[2,5],[3,9],[1,15]]`:

| lần bấm | nút | thời điểm | thời điểm trước | thời gian bấm |
|---|---|---|---|---|
| 1 | 1 | 2 | 0 (bắt đầu) | 2 |
| 2 | 2 | 5 | 2 | 3 |
| 3 | 3 | 9 | 5 | 4 |
| 4 | 1 | 15 | 9 | **6** |

Lâu nhất là `6`, thuộc nút `1` ⇒ đáp án `1`. Lưu ý nút `1` xuất hiện **hai lần** (thời gian `2` và `6`); ta chỉ cần lần **lâu nhất** của nó.

Một ví dụ **hoà** (tự dựng): `events = [[3,4],[1,8],[2,12],[1,14]]`. Thời gian: nút `3`: `4`; nút `1`: `8 - 4 = 4`; nút `2`: `12 - 8 = 4`; nút `1`: `14 - 12 = 2`. Ba nút `3, 1, 2` cùng `4` ⇒ chọn chỉ số **nhỏ nhất** là `1`.

In [7]:
def show_durations(events):
    prev = 0
    print("events =", events)
    for k, (idx, t) in enumerate(events):
        d = t - prev
        print(f"  lần {k + 1}: nút {idx:>2}, lúc {t:>3}, trước đó {prev:>3} -> thời gian {d}")
        prev = t

show_durations([[1, 2], [2, 5], [3, 9], [1, 15]])
show_durations([[3, 4], [1, 8], [2, 12], [1, 14]])      # tự dựng: có hoà

events = [[1, 2], [2, 5], [3, 9], [1, 15]]
  lần 1: nút  1, lúc   2, trước đó   0 -> thời gian 2
  lần 2: nút  2, lúc   5, trước đó   2 -> thời gian 3
  lần 3: nút  3, lúc   9, trước đó   5 -> thời gian 4
  lần 4: nút  1, lúc  15, trước đó   9 -> thời gian 6
events = [[3, 4], [1, 8], [2, 12], [1, 14]]
  lần 1: nút  3, lúc   4, trước đó   0 -> thời gian 4
  lần 2: nút  1, lúc   8, trước đó   4 -> thời gian 4
  lần 3: nút  2, lúc  12, trước đó   8 -> thời gian 4
  lần 4: nút  1, lúc  14, trước đó  12 -> thời gian 2


## 5. Cách 1 — Brute Force / baseline: lập danh sách thời gian rồi tìm lớn nhất

**Ý tưởng.** Làm hai bước tách biệt, rất dễ hiểu:

1. **Bước 1:** lập danh sách `durations[k]` = thời gian của lần bấm thứ `k` (`events[k][1]` trừ thời điểm trước).
2. **Bước 2:** tìm `longest = max(durations)`, rồi trong các lần bấm có `durations[k] == longest`, lấy **chỉ số nút nhỏ nhất**.

In [8]:
def button_two_pass(events: List[List[int]]) -> int:
    # Bước 1: danh sách thời gian của từng lần bấm
    durations = []
    prev = 0
    for idx, t in events:
        durations.append(t - prev)
        prev = t

    # Bước 2: lớn nhất, rồi chọn nút có chỉ số nhỏ nhất trong số hoà
    longest = max(durations)
    answer = None
    for k in range(len(events)):
        if durations[k] == longest:
            if answer is None or events[k][0] < answer:
                answer = events[k][0]
    return answer

for events, expected in examples:
    print(button_two_pass(events), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
10 (kỳ vọng 10 )


**Độ phức tạp (Cách 1).** Ba lượt duyệt (`n` lần bấm): **Time `O(n)`**; **Space `O(n)`** cho `durations`.

**Nút thắt.** Danh sách `durations` chỉ để **tìm lớn nhất rồi duyệt lại**. Ta không cần giữ mọi thời gian; chỉ cần biết **kỷ lục hiện tại** khi duyệt. Một lượt duyệt, `O(1)` bộ nhớ là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ kỷ lục khi duyệt.** Lập hai biến: `t` (thời gian bấm lâu nhất đến giờ) và `ans` (nút tương ứng). Với mỗi lần bấm `(idx, time)`:

1. `d = time - prev`;
2. nếu `d > t` **hoặc** (`d == t` **và** `idx < ans`) thì `t, ans = d, idx`;
3. `prev = time`.

Khởi tạo bằng **lần bấm đầu tiên**: `ans, t = events[0]` (vì thời gian của nó chính là `time`). Sau đó duyệt từ lần thứ hai trở đi.

Vì sao hợp lý: với mọi tập các lần bấm đã xét, `(t, ans)` luôn là kết quả đúng (lâu nhất, hoà chọn chỉ số nhỏ nhất) — một **bất biến** của vòng lặp.

In [9]:
# Demo: cập nhật kỷ lục với events = [[3,4],[1,8],[2,12],[1,14]]  (có hoà)
events = [[3, 4], [1, 8], [2, 12], [1, 14]]
ans, t = events[0]
print(f"khởi tạo: ans={ans}, t={t}")
prev = t
for idx, time in events[1:]:
    d = time - prev
    nhan = d > t or (d == t and idx < ans)
    if nhan:
        ans, t = idx, d
    print(f"nút {idx}: d={d} -> {'nhận' if nhan else 'bỏ'} | ans={ans}, t={t}")
    prev = time

khởi tạo: ans=3, t=4
nút 1: d=4 -> nhận | ans=1, t=4
nút 2: d=4 -> bỏ | ans=1, t=4
nút 1: d=2 -> bỏ | ans=1, t=4


## 7. Các cách tiếp theo

### Cách 2 — Gom theo nút bằng `dict`, rồi chọn

**Giải thích từng đoạn.**

1. Tạo `best = {}`: `best[idx]` = thời gian bấm **lâu nhất** của nút `idx`.
2. Duyệt các lần bấm; với mỗi lần tính `d` và cập nhật `best[idx] = max(best.get(idx, 0), d)`.
3. Chọn nút có `best[idx]` lớn nhất; hoà thì chỉ số nhỏ nhất. Dùng `max(best, key=lambda b: (best[b], -b))`.

In [10]:
def button_dict(events: List[List[int]]) -> int:
    best = {}                        # nút -> thời gian lâu nhất
    prev = 0
    for idx, t in events:
        d = t - prev
        best[idx] = max(best.get(idx, 0), d)
        prev = t
    return max(best, key=lambda b: (best[b], -b))

for events, expected in examples:
    print(button_dict(events), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
10 (kỳ vọng 10 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from typing import List

class SolutionDict:
    def buttonWithLongestTime(self, events: List[List[int]]) -> int:
        best = {}
        prev = 0
        for idx, t in events:
            d = t - prev
            best[idx] = max(best.get(idx, 0), d)
            prev = t
        return max(best, key=lambda b: (best[b], -b))

print(SolutionDict().buttonWithLongestTime([[10, 5], [1, 7]]))   # 10

10


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(m)`** với `m` là số nút khác nhau (`<= n`). Lợi ích: có luôn bảng "nút → thời gian lâu nhất" cho các câu hỏi mở rộng.

### Cách 3 — Một lượt duyệt, giữ `ans` và `t` (lời giải chính thức)

**Giải thích từng đoạn.**

1. `ans, t = events[0]` — lần đầu: nút `events[0][0]`, thời gian `events[0][1]`.
2. `for (_, t1), (i, t2) in pairwise(events):` — mỗi lần là một cặp liền kề; `t1` là thời điểm **trước**, `t2` thời điểm **hiện tại**, `i` là nút hiện tại.
3. `d = t2 - t1`; `if d > t or (d == t and i < ans): ans, t = i, d`.
4. `return ans`.

In [12]:
from itertools import pairwise

def button_onepass(events: List[List[int]]) -> int:
    ans, t = events[0]
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d > t or (d == t and i < ans):
            ans, t = i, d
    return ans

for events, expected in examples:
    print(button_onepass(events), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
10 (kỳ vọng 10 )


In [13]:
# Code nộp LeetCode cho Cách 3
from itertools import pairwise
from typing import List

class SolutionOnePass:
    def buttonWithLongestTime(self, events: List[List[int]]) -> int:
        ans, t = events[0]
        for (_, t1), (i, t2) in pairwise(events):
            d = t2 - t1
            if d > t or (d == t and i < ans):
                ans, t = i, d
        return ans

print(SolutionOnePass().buttonWithLongestTime([[1, 2], [2, 5], [3, 9], [1, 15]]))   # 1

1


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — `max` với khoá `(thời gian, -chỉ số)`

**Giải thích.** Lập generator `(d, -idx)` cho mỗi lần bấm rồi lấy `max`. Vì tuple so sánh từ trái sang phải, `max` chọn **thời gian lớn nhất**; nếu hoà thì chọn `-idx` lớn nhất, tức `idx` **nhỏ nhất**. Cuối cùng đổi dấu lại để lấy `idx`.

Để tính `d` ta cần `prev`; có thể dùng `zip` giữa danh sách thời điểm và danh sách thời điểm trước: `prev_times = [0] + [t for _, t in events[:-1]]`.

In [14]:
def button_max_key(events: List[List[int]]) -> int:
    prev_times = [0] + [t for _, t in events[:-1]]        # thời điểm trước của mỗi lần
    d, neg_idx = max((t - p, -idx) for (idx, t), p in zip(events, prev_times))
    return -neg_idx

for events, expected in examples:
    print(button_max_key(events), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
10 (kỳ vọng 10 )


In [15]:
# Code nộp LeetCode cho Cách 4
from typing import List

class SolutionMax:
    def buttonWithLongestTime(self, events: List[List[int]]) -> int:
        prev_times = [0] + [t for _, t in events[:-1]]
        _, neg_idx = max((t - p, -idx) for (idx, t), p in zip(events, prev_times))
        return -neg_idx

print(SolutionMax().buttonWithLongestTime([[3, 4], [1, 8], [2, 12], [1, 14]]))   # 1 (hoà ba nút, chọn nhỏ nhất)

1


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (mảng `prev_times`). Ngắn gọn, nhưng đòi hỏi quen với so sánh tuple. Cả bốn cách cho cùng kết quả.

## 8. TRACE từng bước

Trace **Cách 3** với `events = [[3,4],[1,8],[2,12],[1,14]]` (có hoà). Cột `ans, t` là sau bước đó.

| bước | `(i, t2)` | `t1` | `d = t2 - t1` | so với `t` | hành động | `ans` | `t` |
|---|---|---|---|---|---|---|---|
| khởi tạo | — | — | — | — | `ans, t = 3, 4` | 3 | 4 |
| 1 | (1, 8) | 4 | 4 | `d == t` và `1 < 3` | **nhận** | 1 | 4 |
| 2 | (2, 12) | 8 | 4 | `d == t` nhưng `2 < 1` sai | bỏ | 1 | 4 |
| 3 | (1, 14) | 12 | 2 | `d < t` | bỏ | 1 | 4 |

Kết quả `1`. Bước 1 và bước 2 cho thấy cơ chế hoà: nút `1` thay nút `3` (nhỏ hơn), nút `2` **không** thay được nút `1`.

In [16]:
from itertools import pairwise

def trace_button(events):
    ans, t = events[0]
    print(f"events = {events}")
    print(f"khởi tạo: ans={ans}, t={t}")
    for k, ((_, t1), (i, t2)) in enumerate(pairwise(events), start=1):
        d = t2 - t1
        if d > t:
            action = f"d={d} > t={t}: nhận"
            ans, t = i, d
        elif d == t and i < ans:
            action = f"d={d} == t và i={i} < ans={ans}: nhận (hoà, chỉ số nhỏ hơn)"
            ans, t = i, d
        elif d == t:
            action = f"d={d} == t nhưng i={i} >= ans={ans}: bỏ"
        else:
            action = f"d={d} < t={t}: bỏ"
        print(f"bước {k}: nút {i}, {t2} - {t1} = {d} -> {action} | ans={ans}, t={t}")
    return ans

trace_button([[3, 4], [1, 8], [2, 12], [1, 14]])
trace_button([[1, 2], [2, 5], [3, 9], [1, 15]])
trace_button([[10, 5], [1, 7]])

events = [[3, 4], [1, 8], [2, 12], [1, 14]]
khởi tạo: ans=3, t=4
bước 1: nút 1, 8 - 4 = 4 -> d=4 == t và i=1 < ans=3: nhận (hoà, chỉ số nhỏ hơn) | ans=1, t=4
bước 2: nút 2, 12 - 8 = 4 -> d=4 == t nhưng i=2 >= ans=1: bỏ | ans=1, t=4
bước 3: nút 1, 14 - 12 = 2 -> d=2 < t=4: bỏ | ans=1, t=4
events = [[1, 2], [2, 5], [3, 9], [1, 15]]
khởi tạo: ans=1, t=2
bước 1: nút 2, 5 - 2 = 3 -> d=3 > t=2: nhận | ans=2, t=3
bước 2: nút 3, 9 - 5 = 4 -> d=4 > t=3: nhận | ans=3, t=4
bước 3: nút 1, 15 - 9 = 6 -> d=6 > t=4: nhận | ans=1, t=6
events = [[10, 5], [1, 7]]
khởi tạo: ans=10, t=5
bước 1: nút 1, 7 - 5 = 2 -> d=2 < t=5: bỏ | ans=10, t=5


10

## 9. Code nộp LeetCode

Ta nộp **Cách 3** (một lượt, `O(1)` bộ nhớ). Với phiên bản không dùng `pairwise` (nếu môi trường cũ hơn Python 3.10), có thể viết vòng `for k in range(1, len(events))` — cũng cho kết quả giống hệt.

In [17]:
from itertools import pairwise
from typing import List

class Solution:
    def buttonWithLongestTime(self, events: List[List[int]]) -> int:
        ans, t = events[0]                      # lần bấm đầu: thời gian = chính time
        for (_, t1), (i, t2) in pairwise(events):
            d = t2 - t1                         # thời gian bấm của nút i
            if d > t or (d == t and i < ans):   # lâu hơn, hoặc hoà nhưng chỉ số nhỏ hơn
                ans, t = i, d
        return ans


for events, expected in examples:
    got = Solution().buttonWithLongestTime(events)
    print(events, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[1, 2], [2, 5], [3, 9], [1, 15]] -> 1 OK
[[10, 5], [1, 7]] -> 10 OK


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý `k` lần bấm đầu tiên (`k >= 1`), `t` bằng thời gian lâu nhất trong `k` lần đó, và `ans` là **chỉ số nút nhỏ nhất** trong các lần bấm đạt thời gian `t`.

**Khởi tạo (`k = 1`).** Chỉ có lần bấm đầu tiên, thời gian của nó bằng `events[0][1]`, nút là `events[0][0]` ⇒ bất biến đúng.

**Bước quy nạp.** Thêm lần bấm `(i, d)`:
- `d > t`: lần mới lâu hơn hẳn mọi lần trước ⇒ kỷ lục mới `t = d`, và nút duy nhất đạt kỷ lục là `i` ⇒ `ans = i`.
- `d == t`: tập các lần bấm đạt kỷ lục được **mở rộng** thêm nút `i`; nút nhỏ nhất của tập mới là `min(ans, i)` ⇒ cập nhật khi `i < ans`.
- `d < t`: không đổi.

**Kết luận.** Hết vòng lặp, `ans` là nút có lần bấm lâu nhất, hoà thì chỉ số nhỏ nhất. Việc một nút bấm **nhiều lần** không ảnh hưởng: mỗi lần là một "ứng viên" riêng, và nút chỉ cần **một lần** đạt kỷ lục.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(events)`, `m` là số nút phân biệt.

- **Cách 1 (hai lượt + danh sách):** Time `O(n)`, Space `O(n)`.
- **Cách 2 (dict theo nút):** Time `O(n)`, Space `O(m)`.
- **Cách 3 (một lượt, `ans`/`t`):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (`max` với khoá tuple):** Time `O(n)`, Space `O(n)` (do dựng `prev_times`).

Mọi cách `O(n)` đều tối ưu thời gian (phải xem mọi sự kiện); Cách 3 tối ưu cả bộ nhớ. Với `n <= 1000` mọi cách đều rất nhanh.

## 12. Bộ test

### 12.1. Ví dụ của đề và các trường hợp biên

In [18]:
impls = {
    "two_pass": button_two_pass,
    "dict":     lambda e: SolutionDict().buttonWithLongestTime(e),
    "onepass":  lambda e: Solution().buttonWithLongestTime(e),
    "max_key":  lambda e: SolutionMax().buttonWithLongestTime(e),
}

cases = [
    ([[1, 2], [2, 5], [3, 9], [1, 15]], 1),          # ví dụ 1
    ([[10, 5], [1, 7]], 10),                         # ví dụ 2
    ([[5, 7]], 5),                                   # tự dựng: một sự kiện
    ([[3, 4], [1, 8], [2, 12], [1, 14]], 1),         # tự dựng: hoà ba nút -> chỉ số nhỏ nhất
    ([[2, 3], [2, 10]], 2),                          # tự dựng: cùng một nút bấm hai lần
    ([[7, 1], [4, 2], [9, 3]], 4),                   # tự dựng: tất cả thời gian bằng 1 -> nút nhỏ nhất (4)
    ([[1, 3], [2, 8], [1, 10], [1, 12]], 2),         # tự dựng: nút 1 tổng thời gian lớn nhưng lần lâu nhất là của nút 2
    ([[100000, 100000]], 100000),                    # tự dựng: giá trị lớn nhất
    ([[1, 1], [2, 100000]], 2),                      # tự dựng: khoảng lớn ở lần sau
    ([[9, 2], [8, 4], [7, 6], [6, 8]], 6),           # tự dựng: hoà hoàn toàn -> chỉ số nhỏ nhất xuất hiện cuối
]

for events, exp in cases:
    for name, f in impls.items():
        got = f(events)
        assert got == exp, (name, events, got, exp)
print("12.1 OK:", len(cases), "test x", len(impls), "cách")

12.1 OK: 10 test x 4 cách


### 12.2. Phản ví dụ cho các cách làm sai

Ta viết các bản sai có chủ đích rồi tìm input làm lộ lỗi.

In [19]:
def wrong_skip_first(events):
    """Sai: bỏ lần bấm đầu tiên (coi thời gian = 0)."""
    ans, t = events[0][0], 0
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d > t or (d == t and i < ans):
            ans, t = i, d
    return ans

def wrong_use_time(events):
    """Sai: dùng time thay vì hiệu."""
    return max(events, key=lambda e: (e[1], -e[0]))[0]

def wrong_no_tie(events):
    """Sai: hoà thì giữ nút gặp trước (không xét chỉ số nhỏ hơn)."""
    ans, t = events[0]
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d > t:
            ans, t = i, d
    return ans

def wrong_ge(events):
    """Sai: dùng >= nên hoà chọn nút xuất hiện SAU cùng."""
    ans, t = events[0]
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d >= t:
            ans, t = i, d
    return ans

def wrong_position(events):
    """Sai: trả vị trí trong danh sách thay vì chỉ số nút."""
    best_k, t = 0, events[0][1]
    for k in range(1, len(events)):
        d = events[k][1] - events[k - 1][1]
        if d > t:
            best_k, t = k, d
    return best_k

def wrong_sum(events):
    """Sai: cộng dồn thời gian của cùng một nút."""
    total, prev = {}, 0
    for idx, t in events:
        total[idx] = total.get(idx, 0) + (t - prev)
        prev = t
    return max(total, key=lambda b: (total[b], -b))

print("skip_first :", wrong_skip_first([[10, 5], [1, 7]]), "(đúng: 10)")
print("use_time   :", wrong_use_time([[1, 2], [2, 5], [3, 9], [1, 15]]), "(đúng: 1) | ví dụ bắt lỗi:", wrong_use_time([[5, 7], [2, 8]]), "(đúng: 5)")
print("no_tie     :", wrong_no_tie([[3, 4], [1, 8], [2, 12], [1, 14]]), "(đúng: 1)")
print("ge         :", wrong_ge([[9, 2], [8, 4], [7, 6], [6, 8]]), "(đúng: 6)")
print("position   :", wrong_position([[10, 5], [1, 7]]), "(đúng: 10)  /", wrong_position([[1, 2], [2, 5], [3, 9], [1, 15]]), "(đúng: 1)")
print("sum        :", wrong_sum([[1, 3], [2, 8], [1, 10], [1, 12]]), "(đúng: 2)")

assert wrong_skip_first([[10, 5], [1, 7]]) == 1
assert wrong_no_tie([[3, 4], [1, 8], [2, 12], [1, 14]]) == 3
assert wrong_ge([[9, 2], [8, 4], [7, 6], [6, 8]]) == 6      # input này KHÔNG bắt được lỗi >= (xem ghi chú dưới)
assert wrong_use_time([[5, 7], [2, 8]]) == 2
assert wrong_position([[1, 2], [2, 5], [3, 9], [1, 15]]) == 3
assert wrong_sum([[1, 3], [2, 8], [1, 10], [1, 12]]) == 1
print("12.2 OK: các bản sai (trừ >=, xem ô sau) đều bị lộ")

skip_first : 1 (đúng: 10)
use_time   : 1 (đúng: 1) | ví dụ bắt lỗi: 2 (đúng: 5)
no_tie     : 3 (đúng: 1)
ge         : 6 (đúng: 6)
position   : 0 (đúng: 10)  / 3 (đúng: 1)
sum        : 1 (đúng: 2)
12.2 OK: các bản sai (trừ >=, xem ô sau) đều bị lộ


Lưu ý: ở bản `wrong_ge` với `[[9,2],[8,4],[7,6],[6,8]]` (mọi thời gian bằng `2`), nút xuất hiện sau cùng là `6` — **trùng** với đáp án đúng nên input đó **không** bắt được lỗi. Cần input mà nút nhỏ nhất **không** ở cuối, như `[[3,4],[1,8],[2,12]]` (đúng `1`, bản `>=` trả `2`).

In [20]:
print(wrong_ge([[3, 4], [1, 8], [2, 12]]), "(đúng: 1)")
assert wrong_ge([[3, 4], [1, 8], [2, 12]]) == 2
assert Solution().buttonWithLongestTime([[3, 4], [1, 8], [2, 12]]) == 1
print("OK: input này bắt được lỗi >=")

2 (đúng: 1)
OK: input này bắt được lỗi >=


### 12.3. Đối chiếu ngẫu nhiên (có seed) với một oracle độc lập

**Oracle độc lập** tính theo định nghĩa: lập danh sách `(thời gian, nút)` cho mọi lần bấm, sắp xếp rồi lấy phần tử tốt nhất (thời gian giảm dần, nút tăng dần). Ta so bốn cách với oracle trên input ngẫu nhiên đúng ràng buộc: `1 <= n <= 1000`, `1 <= index, time <= 10^5`, `time` tăng dần.

In [21]:
import random

def oracle(events):
    items = []
    prev = 0
    for idx, t in events:
        items.append((t - prev, idx))
        prev = t
    items.sort(key=lambda p: (-p[0], p[1]))     # lâu nhất trước, hoà thì nút nhỏ trước
    return items[0][1]

def gen_events(n, max_idx, max_gap):
    times, cur = [], 0
    for _ in range(n):
        cur += random.randint(1, max_gap)
        times.append(cur)
    return [[random.randint(1, max_idx), t] for t in times]

random.seed(3386)
count = 0
for _ in range(1500):
    n = random.randint(1, 10)
    events = gen_events(n, random.choice([3, 10, 100000]), random.choice([2, 5, 1000]))
    exp = oracle(events)
    for name, f in impls.items():
        assert f(events) == exp, (name, events)
    count += 1

# n lớn đúng ràng buộc
for _ in range(100):
    n = 1000
    events = gen_events(n, random.choice([5, 100000]), 99)      # time tối đa 1000 * 99 < 10^5
    assert events[-1][1] <= 100000
    exp = oracle(events)
    for name, f in impls.items():
        assert f(events) == exp
    count += 1
print("12.3 OK:", count, "input ngẫu nhiên, 4 cách khớp oracle")

12.3 OK: 1600 input ngẫu nhiên, 4 cách khớp oracle


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ sót lần bấm đầu tiên

Thời gian của lần đầu **không phải 0** mà chính là `time` của nó (tính từ mốc 0). Nếu bỏ qua, ví dụ 2 trả `1` thay vì `10`.

In [22]:
# SAI
def sai_1(events):
    ans, t = events[0][0], 0                # <-- sai: lần đầu coi là 0
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d > t or (d == t and i < ans):
            ans, t = i, d
    return ans

print(sai_1([[10, 5], [1, 7]]))             # 1 (sai; đúng là 10)
print(Solution().buttonWithLongestTime([[10, 5], [1, 7]]))   # 10

1
10


### Lỗi 2 — Dùng `time` thay vì hiệu thời gian

`time` là **thời điểm**, không phải thời lượng. Với `[[5,7],[2,8]]`: nút `5` giữ `7`, nút `2` giữ `8 - 7 = 1` ⇒ đáp án `5`; nếu so `time` ta chọn nút `2` (vì `8 > 7`).

In [23]:
def sai_2(events):
    return max(events, key=lambda e: (e[1], -e[0]))[0]    # <-- sai: so thời điểm

print(sai_2([[5, 7], [2, 8]]))                             # 2 (sai)
print(Solution().buttonWithLongestTime([[5, 7], [2, 8]]))  # 5 (đúng)

2
5


### Lỗi 3 — Quên (hoặc làm sai) luật hoà

Đề yêu cầu hoà thì chọn **chỉ số nhỏ nhất**. Hai lỗi thường gặp: (a) không xét hoà (giữ nút gặp trước), (b) dùng `>=` (lấy nút gặp **sau** cùng).

In [24]:
events = [[3, 4], [1, 8], [2, 12]]          # mọi thời gian bằng 4

def sai_3a(events):                          # không xét hoà
    ans, t = events[0]
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d > t:
            ans, t = i, d
    return ans

def sai_3b(events):                          # dùng >=
    ans, t = events[0]
    for (_, t1), (i, t2) in pairwise(events):
        d = t2 - t1
        if d >= t:
            ans, t = i, d
    return ans

print(sai_3a(events), sai_3b(events), Solution().buttonWithLongestTime(events))   # 3 2 1

3 2 1


### Lỗi 4 — Trả vị trí trong danh sách thay vì chỉ số nút

Câu hỏi là **`index` của nút** (`events[k][0]`), không phải `k`. Hai thứ này khác nhau (ví dụ 2: nút `10` nằm ở vị trí `0`).

In [25]:
def sai_4(events):
    best_k, t = 0, events[0][1]
    for k in range(1, len(events)):
        d = events[k][1] - events[k - 1][1]
        if d > t:
            best_k, t = k, d
    return best_k                                # <-- sai: trả vị trí

print(sai_4([[10, 5], [1, 7]]))                   # 0 (sai; đúng là 10)
print(sai_4([[1, 2], [2, 5], [3, 9], [1, 15]]))   # 3 (sai; đúng là 1)
print(Solution().buttonWithLongestTime([[10, 5], [1, 7]]))   # 10

0
3
10


### Lỗi 5 — Cộng dồn thời gian của cùng một nút

Đề so sánh **thời gian của một lần bấm**, không phải tổng thời gian của nút. Với `[[1,3],[2,8],[1,10],[1,12]]`: nút `1` có các lần `3, 2, 2` (tổng `7`), nút `2` có lần `5`. Lần lâu nhất là `5` của nút `2`.

In [26]:
def sai_5(events):
    total, prev = {}, 0
    for idx, t in events:
        total[idx] = total.get(idx, 0) + (t - prev)    # <-- sai: cộng dồn
        prev = t
    return max(total, key=lambda b: (total[b], -b))

print(sai_5([[1, 3], [2, 8], [1, 10], [1, 12]]))              # 1 (sai)
print(Solution().buttonWithLongestTime([[1, 3], [2, 8], [1, 10], [1, 12]]))   # 2 (đúng)

1
2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**giữ giá trị lớn nhất / nhỏ nhất khi duyệt, kèm vị trí**" ⇒ mẫu **"kỷ lục + người giữ kỷ lục"**: `(best, who)` cập nhật khi tốt hơn.
- Có **luật hoà** ("nhỏ nhất chỉ số") ⇒ đặt điều kiện `d > t or (d == t and idx < ans)`, hoặc đưa vào khoá `(d, -idx)`.
- Khoảng cách giữa các sự kiện **liên tiếp** ⇒ giữ `prev` hoặc dùng `pairwise`.
- Cẩn thận **lần đầu tiên** (không có "lần trước"): khởi tạo bằng sự kiện đầu hoặc đặt `prev = 0` đúng như mô tả đề.
- Đọc kỹ **đáp án là gì**: chỉ số nút (`events[k][0]`) hay vị trí `k`; thời gian hay nút.
- Dữ liệu đã được **sắp xếp** theo thời gian ⇒ không cần sort lại; hiệu liên tiếp luôn dương.
- Khi có thể: kiểm bằng một **oracle** viết theo định nghĩa (sắp xếp mọi ứng viên) cho input ngẫu nhiên.

## 15. Biến thể của bài

- **Nút có thời gian bấm ngắn nhất:** đổi `d > t` thành `d < t` (và luật hoà tương tự) — bài tập mở rộng.
- **Tổng thời gian bấm của mỗi nút** (cộng dồn): dùng `dict` với `+=` (đó **không** phải bài này nhưng là biến thể hay gặp).
- **Trả về mọi nút hoà nhau ở vị trí lâu nhất** (bài tập 5).
- **Thời gian lâu nhất thứ hai** hoặc `k` lớn nhất: dùng `sorted`/`heapq`.
- **Sự kiện chưa sắp xếp:** cần `sorted(events, key=lambda e: e[1])` trước.
- **Khoảng cách tối đa giữa hai sự kiện liên tiếp bất kỳ:** cùng mẫu `pairwise`, không cần nút.
- **Nhiều "người chơi" (mỗi người một dãy sự kiện):** áp dụng thuật toán cho từng người rồi tổng hợp.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai lượt + danh sách | Lập `durations`, tìm max, chọn chỉ số nhỏ nhất | `O(n)` | `O(n)` | Dễ hiểu nhất; để học và kiểm chứng |
| 2. `dict` theo nút | `best[nút] = max(...)`, chọn bằng khoá | `O(n)` | `O(m)` | Khi cần bảng thời gian lâu nhất của từng nút |
| 3. Một lượt `ans`/`t` | Giữ kỷ lục, xét hoà | `O(n)` | `O(1)` | **Cách nộp chính**; tối ưu bộ nhớ |
| 4. `max` với khoá tuple | `(d, -idx)` | `O(n)` | `O(n)` | Khi muốn code ngắn, quen tuple |

Mọi cách cùng đáp án (đã đối chiếu oracle ở mục 12.3).

## 17. 5 bài tập vận dụng (dễ → khó)

Các cell TODO **không được gọi khi chạy notebook** — hãy tự điền rồi tự chạy. Đáp án ở mục 19.

### Bài 1 — Đoán output

Đoán kết quả `bai1()` (không chạy).

In [27]:
def bai1():
    events = [[4, 1], [2, 4], [9, 6], [2, 10]]
    prev, out = 0, []
    for idx, t in events:
        out.append((idx, t - prev))
        prev = t
    return out, max(out, key=lambda p: (p[1], -p[0]))[0]

# Hãy đoán rồi mới chạy:  print(bai1())

### Bài 2 — Tính tay

Với `events = [[5, 3], [7, 6], [1, 9]]`: tính thời gian bấm của từng lần, rồi xác định nút lâu nhất (chú ý luật hoà). Điền hai biến dưới đây (không chạy hàm để lấy đáp án).

In [28]:
# TODO: tự tính tay rồi điền
bai2_durations = None     # danh sách 3 thời gian
bai2_ans = None           # chỉ số nút

### Bài 3 — Điền TODO

Hoàn thành lời giải một lượt (Cách 3) **không dùng `pairwise`**, bằng vòng `for k in range(1, len(events))`.

In [29]:
def bai3(events):
    ans, t = events[0]
    for k in range(1, len(events)):
        # TODO: i = events[k][0]; d = events[k][1] - events[k - 1][1]
        ...
        # TODO: nếu d > t hoặc (d == t và i < ans) thì cập nhật ans, t
        ...
    return ans

### Bài 4 — Tự viết lại

Viết `bai4(events)` trả về **từ điển** `{nút: thời gian bấm lâu nhất của nút đó}`. Ví dụ `bai4([[1,2],[2,5],[3,9],[1,15]]) = {1: 6, 2: 3, 3: 4}`.

In [30]:
def bai4(events):
    # TODO: tự viết
    ...

### Bài 5 — Biến thể / mini challenge

Viết `bai5(events)` trả về **danh sách (tăng dần) mọi nút** có lần bấm đạt thời gian lâu nhất. Ví dụ `bai5([[3,4],[1,8],[2,12],[1,14]]) = [1, 2, 3]`, `bai5([[1,2],[2,5],[3,9],[1,15]]) = [1]`.

In [31]:
def bai5(events):
    # TODO: tự viết
    ...

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.**
- Mức 1: tính hiệu liên tiếp: `1 - 0`, `4 - 1`, `6 - 4`, `10 - 6`.
- Mức 2: thời gian `1, 3, 2, 4` ứng với nút `4, 2, 9, 2`.
- Mức 3: lâu nhất `4` của nút `2` ⇒ kết quả `([(4, 1), (2, 3), (9, 2), (2, 4)], 2)`.

**Bài 2.**
- Mức 1: lần đầu thời gian bằng `3`.
- Mức 2: `6 - 3 = 3`, `9 - 6 = 3` — tất cả bằng nhau.
- Mức 3: hoà ba nút `5, 7, 1` ⇒ chọn chỉ số nhỏ nhất `1`.

**Bài 3.**
- Mức 1: `i, t2 = events[k]`; thời điểm trước là `events[k - 1][1]`.
- Mức 2: `d = t2 - events[k - 1][1]`.
- Mức 3: `if d > t or (d == t and i < ans): ans, t = i, d`.

**Bài 4.**
- Mức 1: dùng `dict`, duyệt kèm `prev`.
- Mức 2: `best[idx] = max(best.get(idx, 0), d)`.
- Mức 3: nhớ cập nhật `prev = t` sau mỗi lần; bắt đầu `prev = 0`.

**Bài 5.**
- Mức 1: tính danh sách `(d, idx)` cho mọi lần bấm.
- Mức 2: `longest = max(d)`; lấy các `idx` có `d == longest`.
- Mức 3: dùng `sorted(set(...))` để loại trùng và sắp tăng dần.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều được kiểm tra bằng `assert` (bài 3, 4, 5 còn đối chiếu với oracle trên input ngẫu nhiên).

In [32]:
# --- Bài 1 ---
assert bai1() == ([(4, 1), (2, 3), (9, 2), (2, 4)], 2)
print("Bài 1:", bai1())

# --- Bài 2 ---
bai2_durations_da = [3, 3, 3]
bai2_ans_da = 1
assert Solution().buttonWithLongestTime([[5, 3], [7, 6], [1, 9]]) == bai2_ans_da
assert oracle([[5, 3], [7, 6], [1, 9]]) == bai2_ans_da
print("Bài 2:", bai2_durations_da, bai2_ans_da)

# --- Bài 3 ---
def bai3_da(events):
    ans, t = events[0]
    for k in range(1, len(events)):
        i = events[k][0]
        d = events[k][1] - events[k - 1][1]
        if d > t or (d == t and i < ans):
            ans, t = i, d
    return ans

random.seed(3387)
for _ in range(500):
    events = gen_events(random.randint(1, 10), random.choice([3, 10, 100]), random.choice([2, 5, 20]))
    assert bai3_da(events) == oracle(events)
print("Bài 3 OK")

# --- Bài 4 ---
def bai4_da(events):
    best, prev = {}, 0
    for idx, t in events:
        d = t - prev
        best[idx] = max(best.get(idx, 0), d)
        prev = t
    return best

assert bai4_da([[1, 2], [2, 5], [3, 9], [1, 15]]) == {1: 6, 2: 3, 3: 4}
for _ in range(300):
    events = gen_events(random.randint(1, 10), random.choice([3, 10]), random.choice([2, 5, 20]))
    b = bai4_da(events)
    assert min(k for k, v in b.items() if v == max(b.values())) == oracle(events)
print("Bài 4 OK")

# --- Bài 5 ---
def bai5_da(events):
    prev, items = 0, []
    for idx, t in events:
        items.append((t - prev, idx))
        prev = t
    longest = max(d for d, _ in items)
    return sorted({idx for d, idx in items if d == longest})

assert bai5_da([[3, 4], [1, 8], [2, 12], [1, 14]]) == [1, 2, 3]
assert bai5_da([[1, 2], [2, 5], [3, 9], [1, 15]]) == [1]
for _ in range(300):
    events = gen_events(random.randint(1, 10), random.choice([3, 10]), random.choice([2, 5]))
    res = bai5_da(events)
    assert res[0] == oracle(events) and res == sorted(res)
print("Bài 5 OK")

Bài 1: ([(4, 1), (2, 3), (9, 2), (2, 4)], 2)
Bài 2: [3, 3, 3] 1
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (một lượt giữ kỷ lục):**

```python
from itertools import pairwise

class Solution:
    def buttonWithLongestTime(self, events):
        ans, t = events[0]
        for (_, t1), (i, t2) in pairwise(events):
            d = t2 - t1
            if d > t or (d == t and i < ans):
                ans, t = i, d
        return ans
```

**Công thức cần nhớ:** thời gian lần `k` là `time[k] - time[k-1]` (lần đầu: `time[0]`); cập nhật khi `d > t` hoặc `d == t` và chỉ số nhỏ hơn; trả **chỉ số nút**.

**Checklist trước khi submit:**

- [ ] Tên method đúng: `buttonWithLongestTime(self, events)`.
- [ ] Khởi tạo bằng lần bấm đầu: `ans, t = events[0]` (thời gian đầu = `time`, không phải `0`).
- [ ] Thời gian là **hiệu** `t2 - t1`, không phải `time`.
- [ ] Luật hoà: `d == t and i < ans` (không dùng `>=`, không bỏ sót hoà).
- [ ] Trả **chỉ số nút** `events[k][0]`, không phải vị trí `k`.
- [ ] Không cộng dồn thời gian của cùng một nút.
- [ ] Đã thử: một sự kiện, hoà nhiều nút, nút lặp lại, chỉ số nhỏ nhất ở giữa/cuối.